# Generate Synthetic PII dataset

WARNING: 
- This code appends new rows to the existing CSV files.
- The generated data by OpenAI is far from correct or perfect. We have manually checked the dataset as much as possible in our time window.

Could be improved by:
- Using correct more exact methods of generating the PII and using a LLM to generate the sentence around it. For example the IBAN numbers generated by LLM do not pass the Luhn check. 

## General functions

In [1]:
import json
import os

import pandas as pd
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
datagen_model = "gpt-4o-mini"


def get_response(system_prompt, question):
    response = client.chat.completions.create(
        model=datagen_model,
        response_format={"type": "json_object"},
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": question}],
    )
    res = json.loads(response.choices[0].message.content)
    return res

## Load in regular expressions from package

In [ ]:
from gptnl_pii_mappers import COMMON_REGEX, EN_REGEX, NL_REGEX

NL_REGEX.update(COMMON_REGEX)
EN_REGEX.update(COMMON_REGEX)

/home/caspar.meijer/miniconda3/envs/pii_detection/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Generate Dutch Data

### Add NER Categories

In [3]:
ner_categories = {
    "time": {"description": "A specific reference to a generally known or personal date and time.", "name": "PII_Time"},
    "birthday": {"description": "A specific reference to a birthday of a person.", "name": "PII_Birthday"},
    "address": {"description": "A specific reference to a generally known or personal address.", "name": "PII_Address"},
    "person": {"description": "A first and/or last name of an publicly unknown or known person.", "name": "PII_Person"},
    "organisation": {"description": "A name or reference to an organization, publicly known or unknown.", "name": "PII_Organisation"},
}

In [4]:
# def pii_regex_prompt(language, pii_category, information, n_sentences=10):
#     return f"""
#     You generate sentences and labels to test {language} PII detection.

#     I am designing a system for {language} PII detection. To determine its quality, I want to see it's performance on edge cases.
#     I want to check how our models work for this PII category: {pii_category}
#     I wrote a regular expression to detect it:
#     {information}

#     You will help me to get clear what this regex does and does not capture.
#     In order to that, you will generate sentences. Write some sentences that clearly contain the PII {pii_category}, and some that are edge cases but are considered {pii_category} and some that contain something very similar to {pii_category}, but are not exactly.
#     Think about what an edge case could look like for this category, and generate {n_sentences} sentences.
#     Write the sentences with at least 10 words in them and you can differ the topics and context in the sentence greatly.
#     Each sentence should contain at least some form of PII.

#     Return the {language} sentences and a 1 to indicate the sentence contains {pii_category} or a 0 to indicate the sentence does not contain {pii_category}.
#     The output should be a json that looks like this:

#     sentence 1: 1,
#     sentence 2: 0,
#     sentence 3: 0

#     Make sure the each key is only one sentence, and each value is only one number: 0 or 1. The output should be in language {language}!
#     """


# def pii_ner_prompt(language, ner_category, information, n_sentences=10):
#     return f"""
#     You generate sentences and labels to test {language} PII detection.

#     I am designing a system for language PII detection. To determine its quality, I want to see it's performance on edge cases.
#     I want to check how our models work for this PII category: {ner_category}, which is {information}.

#     You will generate sentences. Write some sentences that clearly contain the PII {ner_category}, and some that are edge cases but are considered {ner_category} and some that contain something very similar to {ner_category}, but are not exactly.
#     Think about what an edge case could look like for this category, and generate {n_sentences} sentences.
#     Write the sentences with at least 10 words in them and you can differ the topics and context in the sentence greatly.
#     Each sentence should contain at least some form of PII.

#     Return the language sentences and a 1 to indicate the sentence contains {ner_category} or a 0 to indicate the sentence does not contain {ner_category}.
#     The output should be a json that looks like this:

#     sentence 1: 1,
#     sentence 2: 0,
#     sentence 3: 0

#     Make sure the each key is only one sentence, and each value is only one number: 0 or 1. The output should be in language {language}!
#     """


def pii_regex_prompt(language, pii_category, information, task_instruction):
    information = information["name"][4:]  # Only name
    return f"""
    Generate a sentence in {language} and labels for testing PII detection.

    The aim is to evaluate a PII detection system, focusing particularly on the PII category: {pii_category}.
    The provided information about the PII and the method of detection is: {information}

    **Task Instructions:**
    - {task_instruction}
    - Produce 1 sentence!
    - The PII is either a code, number or other sequence of characters. Think what the {pii_category} actually is and put it in the sentence.

    **Requirements:**
    - Each sentence should be composed of at least 10 words.
    - Contexts and topics should be varied widely to simulate real-world scenarios.

    **Output:**
    - Output as a JSON object with each sentence as a key paired with keys "label: '1' for presence of {pii_category} and '0' for absence, and key "pii_entity" which repeats which pii entity is present in the set. 
    - Produce 1 sentence!

    Example JSON structure (in {language}):
    {{
        "Sentence in {language} #1": {{"label": 0 or 1, "pii_entity":  pii instance }},
        "Sentence in {language} #2": {{"label": 0 or 1, "pii_entity":  pii instance }},
        "Sentence in {language} #3": {{"label": 0 or 1, "pii_entity":  pii instance }}
    }}
    Ensure that the JSON keys are the full sentences in {language}, the values are correctly labeled as integers and the pii_entity is exactly present in the sentence.
    """


def pii_ner_prompt(language, ner_category, information, task_instruction):
    # information = information["name"][4:]  # Only name
    return f"""
    Generate a sentence in {language} and labels for testing PII detection.

    This exercise is to assess a PII detection system utilizing named entity recognition (NER) for the PII category: {ner_category}, described as: {information}.

    **Task Instructions:**
    - {task_instruction}
    - Produce 1 sentence!


    **Requirements:**
    - Sentences must consist of at least 10 words.
    - Ensure diverse contexts and sentence structures to provide variability.
    - Include the PII in each sentence, with focus varying from clear to nuanced.

    **Output:**
    - Output as a JSON object with each sentence as a key paired with keys "label: '1' for presence of {ner_category} and '0' for absence, 
    and key "pii_entity" which repeats which pii entity is present in the set. 
    - Produce 1 sentence!

    Example JSON structure (in {language}):
    {{
        "Sentence in {language} #1": {{"label": 0 or 1, "pii_entity":  pii instance }},
        "Sentence in {language} #2": {{"label": 0 or 1, "pii_entity":  pii instance }},
        "Sentence in {language} #3": {{"label": 0 or 1, "pii_entity":  pii instance }}
    }}
    Ensure that the JSON keys are the full sentences in {language}, the values are correctly labeled as integers and the pii_entity is exactly present in the sentence.
    """


task_instructions = [
    ("True", "Write a sentence where the context suggests that {pii_category} is in the sentence, and {pii_category} is indeed in the sentence. So Label is 1!", 5),
    (
        "False",
        "Write a sentence where the context suggests that {pii_category} is in the sentence, but instead of {pii_category}, some other PII is present in the data that looks like {pii_category}. So label is 0!",
        5,
    ),
    ("Edge", "Write sentences edge case sentences that are tricky for the regex but fall under {pii_category}. Where the label is 1!", 2),
]


# Create Datasets

## Dutch

In [5]:
output_nl = {}  # kan ook initializeren met {}


def generate_output_dictionary(input_dictionary, language, ner=False):
    regex_nl_system_prompt = f"You are a helpful assistant designed to generate edge cases for {language} PII detection."
    output_dictionary = {}
    for pii_category, information in input_dictionary.items():
        output_dictionary[information["name"]] = {"Sentence": [], "Label": [], "Type": [], "Entity": []}
        for task_type, task_instruction, n_sentences in task_instructions:
            for _ in range(n_sentences):
                task_instruction = task_instruction.replace("{pii_category}", pii_category)
                if ner:
                    question = pii_ner_prompt(language, pii_category, information, task_instruction)
                else:
                    question = pii_regex_prompt(language, pii_category, information, task_instruction)
                for i in range(3):  # Validate data types in creation, will retry 3 times until failure.
                    regex_result = get_response(regex_nl_system_prompt, question)

                    if not isinstance(regex_result, dict):
                        print("VALIDATION CHECK (invalid dict):", regex_result)
                        continue

                    if not all(isinstance(key, str) for key in regex_result.keys()):
                        print("VALIDATION CHECK (not str keys):", regex_result)
                        continue

                    subdict = list(regex_result.values())[0]

                    if not isinstance(subdict.get("label"), int):
                        print("VALIDATION CHECK (label not int):", regex_result)
                        continue

                    pii_entity = subdict.get("pii_entity")

                    if not isinstance(pii_entity, str):
                        print("VALIDATION CHECK (pii_entity not str):", regex_result)
                        continue
                    if pii_entity not in list(regex_result.keys())[0]:
                        print("VALIDATION CHECK (pii_entity not substr):", regex_result)
                        continue

                    break
                sentence, label_and_entity = next(iter(regex_result.items()))
                label, entity = label_and_entity.values()
                output_dictionary[information["name"]]["Sentence"].append(sentence)
                output_dictionary[information["name"]]["Label"].append(label)
                output_dictionary[information["name"]]["Entity"].append(entity)
                output_dictionary[information["name"]]["Type"].append(task_type)

        print(information["name"], output_dictionary[information["name"]])
    return output_dictionary


output_nl.update(generate_output_dictionary(NL_REGEX, "Dutch"))
output_nl.update(generate_output_dictionary(ner_categories, "Dutch", ner=True))

PII_DutchBelgianDriversLicenseNumber {'Sentence': ['Toen ik mijn rijbewijs ging vernieuwen, ontdekte ik dat mijn stuurvergunning nummer 1B3C-4567-89D0 verloren was.', 'Mijn rijbewijs nummer is G1 2345 6789 1234 en het is onlangs vernieuwd.', 'Ik moet mijn rijbewijsnummer, 1D23456789, bij de autoriteiten indienen.', 'Graag ontvang ik een kopie van mijn rijbewijs met nummer 1234-AB-56789 voor het aanvragen van een verzekering.', 'Vandaag heb ik mijn rijbewijsnummer gecontroleerd en het is 1234-ABCD-5678', 'Tijdens ons gesprek noemde hij dat zijn identificatienummer 1234-ABCD-5678 was, wat niet zijn rijbewijsnummer is.', 'De code voor jouw inschrijving is 1234-5678-9012, maar dat is geen rijbewijsnummer.', 'Vandaag heb ik mijn belastingnummer 1234.5678.9012 ontvangen, maar ik zoek nog mijn rijbewijsgegevens.', 'Ik heb vorige week mijn bankrekening geopend met het nummer 123456789, maar ik moet mijn rijbewijsnummer nog doorgeven.', 'Een vriend van mij heeft de code 1234-ABCD-5678 op zijn v

## English

In [ ]:
output_en = {}
output_en.update(generate_output_dictionary(EN_REGEX, "English"))
output_en.update(generate_output_dictionary(ner_categories, "English", ner=True))

VALIDATION CHECK (pii_entity not substr): {"I recently applied for a new UK driver's license with the number AB123456C.": {'label': 1, 'pii_entity': 'UKDriversLicense'}}
VALIDATION CHECK (pii_entity not substr): {"After reviewing my application, I realized my UK driver's license number is JX1234567890.": {'label': 1, 'pii_entity': "UK driver's license number JX1234567890"}}
PII_UKDriversLicense {'Sentence': ['I recently had to update my address on my UK driving license, which is A1234567B.', "I need to update my address on my UK driver's license number AB123456C.", "After receiving my application, the DMV issued my UK driver's license with the number AB123456C.", "After checking my paperwork, I realized my UK driver's license number is AB123456C.", "I recently renewed my UK driver's license, which has the number AB123456C", 'The code AB123456C was used for my online grocery order verification.', "The driver's license number is AB123456C, which is a different identifier.", 'I checked my

## Write to files

In [ ]:
def append_to_csv(folder_path, file_name, new_data):
    """Reads a CSV file if it exists, appends new rows, and writes back to the file.
    If the file does not exist, it creates a new CSV with the provided DataFrame.

    Parameters:
    - folder_path (str): Path to folder of CSV file
    - file_name (str): Path to the CSV file.
    - new_data (pd.DataFrame): DataFrame containing new rows to append.

    Returns:
    - None
    """
    os.makedirs(folder_path, exist_ok=True)
    file_path = os.path.join(folder_path, file_name)

    if os.path.exists(file_path):
        # Read the existing data
        existing_data = pd.read_csv(file_path, sep=";")
        # Append new rows
        updated_data = pd.concat([existing_data, new_data], ignore_index=True)
    else:
        # If file doesn't exist, use the new data as the initial content
        updated_data = new_data

    # Write back to the CSV file
    updated_data.to_csv(file_path, index=False, sep=";")


def write_to_csv(language, output_obj):
    for category, test_examples in output_obj.items():
        append_to_csv(language, f"{category}.csv", pd.DataFrame(test_examples))


write_to_csv("Dutch", output_nl)

write_to_csv("English", output_en)